# От API языковой модели к LangChain

**Курс «Разработка и внедрение AI-агентов», МФТИ.**

Весь материал занятия построен вокруг одного бытового вопроса:

> **Поместятся ли 28 студентов в аудитории A-101?**

Человек ответит на него за минуту: посмотрит табличку на двери или спросит в учебной части.
Приложению с языковой моделью для этого нужно пройти заметно более длинный путь. Модель ничего
не знает о нашей аудитории, поэтому приложение должно дать ей доступ к справочнику, выполнить
запрос к справочнику, вернуть результат модели и только потом получить ответ. Этот путь мы и
пройдём — руками, шаг за шагом, без готовых агентов.

### Зачем спускаться на уровень API

На прошлых занятиях модель была одним из узлов графа LangGraph: вы вызывали её и получали ответ.
Сегодня разберём, что происходит внутри этого вызова: какой HTTP-запрос уходит в сеть, где в
ответе лежит текст, как модель просит вызвать функцию и кто эту функцию исполняет. Такие знания
нужны каждый раз, когда агент ведёт себя неожиданно: отлаживать приходится именно на этом уровне.
Фреймворк скрывает детали, пока всё работает, и перестаёт помогать, когда что-то сломалось.

### Маршрут занятия

1. **Подключение к GigaChat.** Установка, ключ авторизации, токен доступа.
2. **Первый запрос.** Два контракта API GigaChat — v2 и v1 — и устройство ответа.
3. **Параметры генерации.** Как модель выбирает слова и почему ответы не повторяются.
4. **Кэш и стоимость.** Какие токены оплачиваются и как их сэкономить.
5. **Другие методы API.** Список моделей, подсчёт токенов, баланс, потоковая выдача.
6. **Зачем модели инструмент.** Вопрос про аудиторию без доступа к данным.
7. **Вызов функции.** Полный обмен «модель → приложение → модель» на GigaChat.
8. **Встроенный инструмент.** Поиск в интернете, который выполняет сам сервер.
9. **GPT2GIGA.** Прокси, который открывает GigaChat для библиотек OpenAI и Anthropic.
10. **OpenAI Chat Completions** — тот же обмен в самом распространённом формате.
11. **OpenAI Responses** — формат с типизированными элементами.
12. **Anthropic Messages** — формат с блоками.
13. **Что уходит в GigaChat.** Как прокси переводит все три формата.
14. **LangChain.** Те же шаги в общих объектах фреймворка.
15. **Практика.**
16. **Итоги.**

### Что нужно знать заранее

Функции и словари Python, формат JSON, общее устройство HTTP-запроса (метод, адрес, заголовки,
тело), роли сообщений в диалоге с моделью. Ячейки выполняются по порядку. Запросы идут к
настоящему API и расходуют токены вашего аккаунта.

<details><summary>Словарь терминов</summary>

- **API** (application programming interface) — способ, которым программа обращается к сервису.
  Здесь это HTTP-запросы к серверу GigaChat.
- **Контракт API** — договорённость о форме запроса и ответа: какие поля передавать, что они значат,
  какие поля придут в ответе.
- **SDK** (software development kit) — библиотека, которая формирует запросы к API и разбирает
  ответы в объекты языка программирования.
- **Тело запроса (body)** — данные, которые передаются в HTTP-запросе; у нас это JSON.
- **Payload** — данные запроса на стороне программы, например словарь Python до превращения в JSON.
- **Сериализация** — превращение объектов программы в передаваемые байты, например словаря в JSON.
- **Токен** (в тексте) — единица, на которые модель делит текст; по токенам считается объём и стоимость.
- **Access token** — временный пропуск для запросов к API, который выдаётся по ключу авторизации.
- **Вызов инструмента (tool call, function call)** — просьба модели выполнить функцию: имя и аргументы.
- **Hook** — функция, которую библиотека вызывает сама в определённый момент работы.
- **Прокси** — промежуточный сервис между клиентом и сервером; у нас переводит форматы запросов.

</details>

Документация и исходники, на которые опирается материал, собраны в [SOURCES.md](SOURCES.md).

## 1. Подключение к GigaChat

### 1.1. Установка

Все зависимости занятия описаны в `pyproject.toml` и закреплены в `uv.lock`. Их ставит
[uv](https://docs.astral.sh/uv/getting-started/installation/) — менеджер проектов Python.
Он же скачает нужную версию Python 3.12, если её нет. Команды одинаковы для macOS, Linux и Windows:

```bash
git clone https://github.com/trashchenkov/mipt_llm_api
cd mipt_llm_api/llm_api_to_langchain
uv sync
cp .env.example .env          # Windows: copy .env.example .env
uv run jupyter lab llm_api_to_langchain.ipynb
```

`uv sync` создаёт в папке виртуальное окружение `.venv` с точными версиями библиотек. В Jupyter
выберите ядро Python из этого окружения — оно подключается по умолчанию, если Jupyter запущен
через `uv run`.

### 1.2. Ключ авторизации

Для работы с GigaChat API нужен **ключ авторизации** (authorization key). Его выдают в личном
кабинете: [developers.sber.ru/studio](https://developers.sber.ru/studio/workspaces/my-space/get/gigachat-api)
→ проект GigaChat API → «Настройки API» → «Получить ключ»
([пошаговая инструкция](https://developers.sber.ru/docs/ru/gigachat/quickstart/ind-create-project)).
Ключ показывается один раз, поэтому сохраните его сразу.

Ключ записывается в файл `.env` в папке занятия:

| Переменная | Назначение |
|---|---|
| `GIGACHAT_CREDENTIALS` | ключ авторизации |
| `GIGACHAT_SCOPE` | область доступа аккаунта; для физических лиц — `GIGACHAT_API_PERS` |
| `GIGACHAT_MODEL` | модель, к которой обращаемся; на занятии — `GigaChat-3-Ultra` |
| `GIGACHAT_VERIFY_SSL_CERTS` | в настройках занятия `False` — проверка сертификатов выключена |

Файл `.env` содержит секрет и в репозиторий не попадает: он указан в `.gitignore`.

Готовность окружения проверяет скрипт — он смотрит версию Python, библиотеки, файл `.env`,
получает токен GigaChat и проверяет прокси:

```bash
uv run tools/check_env.py
```

### 1.3. Как устроена авторизация

У GigaChat два разных секрета, и их важно не путать.

**Ключ авторизации** — долговременный. Технически это строка base64 от пары
`client_id:client_secret` проекта. Им нельзя подписывать запросы к модели: он нужен только
для того, чтобы получить токен.

**Access token** — временный пропуск на 30 минут. Именно он передаётся в каждом запросе к модели
в заголовке `Authorization: Bearer <токен>`. Токен выдаёт отдельный сервер авторизации по
протоколу OAuth 2.0. Запрос за токеном выглядит так (значения заменены заглушками):

```http
POST https://ngw.devices.sberbank.ru:9443/api/v2/oauth
Authorization: Basic <ключ авторизации>
RqUID: 6f0b1291-c7f3-43c6-bb2e-9f3efb2dc98e
Content-Type: application/x-www-form-urlencoded

scope=GIGACHAT_API_PERS
```

- `Authorization: Basic …` — сам ключ авторизации;
- `RqUID` — уникальный идентификатор этого запроса в формате UUID4;
- тело — форма (не JSON) с одним полем `scope`.

Ответ содержит два поля:

```json
{"access_token": "<токен>", "expires_at": 1790597265970}
```

`expires_at` — момент истечения в миллисекундах Unix-времени. Сам токен — строка формата JWE
(зашифрованный JWT): клиент хранит и передаёт его, но прочитать содержимое может только сервер.

**Вручную этот запрос делать не нужно.** SDK GigaChat читает ключ из окружения, сам получает
токен при первом запросе, запоминает его и заранее обновляет перед истечением. Если токен всё же
истёк или неверен, API отвечает `401 Unauthorized`.

### 1.4. Способы авторизации в SDK

SDK поддерживает четыре способа передать доступ. На занятии используется первый; остальные
показаны, чтобы было понятно, в какие поля подставляется каждый вариант.

| Способ | Параметры `GigaChat(...)` | Переменные окружения | Кто обновляет токен |
|---|---|---|---|
| Ключ авторизации | `credentials`, `scope` | `GIGACHAT_CREDENTIALS`, `GIGACHAT_SCOPE` | SDK, автоматически |
| Готовый токен | `access_token` | `GIGACHAT_ACCESS_TOKEN` | никто: после истечения — `401` |
| Логин и пароль | `user`, `password` | `GIGACHAT_USER`, `GIGACHAT_PASSWORD` | SDK (запрос `POST /token`) |
| Клиентский сертификат | `cert_file`, `key_file`, `key_file_password` | `GIGACHAT_CERT_FILE`, `GIGACHAT_KEY_FILE` | вход по сертификату (mTLS) |

```python
# Ключ авторизации (так работает занятие; значения берутся из .env)
client = GigaChat(credentials="<ключ>", scope="GIGACHAT_API_PERS")

# Токен, полученный заранее где-то ещё
client = GigaChat(access_token="<токен>")

# Логин и пароль
client = GigaChat(user="<логин>", password="<пароль>")

# Клиентский сертификат
client = GigaChat(cert_file="client.crt", key_file="client.key")
```

Частая ошибка — вставить в `GIGACHAT_CREDENTIALS` `client_secret` проекта вместо ключа авторизации.
SDK распознаёт это и сообщает: «Invalid credentials format… use only base64 credentials».

### 1.5. Загрузка настроек

Первая ячейка с кодом загружает `.env` в переменные окружения процесса и проверяет, что ключ
и модель заданы. Значения ключей на экран не выводятся — это правило действует на всё занятие:
выводы ноутбука легко случайно опубликовать.

`load_dotenv(..., override=False)` не перезаписывает переменные, уже заданные в окружении.
Если вы экспортировали, например, `GIGACHAT_MODEL` в терминале, будет использовано её значение.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
assert (ROOT / "room_tool.py").is_file(), "Запустите Jupyter из папки llm_api_to_langchain"
load_dotenv(ROOT / ".env", override=False)
for key in ("GIGACHAT_CREDENTIALS", "GIGACHAT_MODEL"):
    print(key, "задан" if os.getenv(key) else "не задан")
assert os.getenv("GIGACHAT_CREDENTIALS"), "Заполните GIGACHAT_CREDENTIALS в .env"
assert os.getenv("GIGACHAT_MODEL"), "Задайте модель в .env"
MODEL = os.environ["GIGACHAT_MODEL"]

### 1.6. Как читать выводы

Ответы API бывают длинными: десятки служебных полей. Чтобы нужное не терялось, в ноутбуке принят
порядок: главные поля печатаются обычным `print`, а полный объект сворачивается в раскрывающийся
блок. Функция `show` делает именно это. Она ничего не запрашивает у API, только оформляет вывод.

Ответы SDK — объекты [Pydantic](https://docs.pydantic.dev/). Метод `model_dump()` превращает такой
объект в обычный словарь, который можно напечатать как JSON.

In [ ]:
import json
from html import escape

from IPython.display import HTML, display


def show(value, title="Полный JSON"):
    if hasattr(value, "model_dump"):
        value = value.model_dump(mode="json", by_alias=True, exclude_none=True)
    text = json.dumps(value, ensure_ascii=False, indent=2)
    display(HTML(f"<details><summary>{escape(title)}</summary><pre>{escape(text)}</pre></details>"))

## 2. Первый запрос: контракты v2 и v1

### 2.1. Два контракта одного API

У GigaChat API два контракта генерации ответа:

- **v2** — `POST https://api.giga.chat/v2/chat/completions`, основной контракт. Сообщение в нём
  состоит из списка частей: текст, ссылки на файлы, результаты функций могут лежать в одном
  сообщении. v2 умеет хранить историю на сервере, запускать встроенные инструменты и подробнее
  описывает результат.
- **v1** — `POST https://api.giga.chat/v1/chat/completions`, предыдущий контракт. Сообщение в нём —
  просто строка. На v1 до сих пор работают многие интеграции, в том числе адаптер LangChain и
  прокси GPT2GIGA, которые мы увидим позже.

В SDK им соответствуют два метода одного клиента:

| Контракт | Метод SDK | Модели запроса (для типизированного кода) |
|---|---|---|
| v2 | `client.chat.create(...)` | `ChatCompletionRequest`, `ChatMessage` |
| v1 | `client.chat(...)` | `Chat`, `Messages` |

Метод `chat()` для v1 сохранён ради совместимости. Он работает и не выдаёт предупреждений,
но новый код разработчики SDK рекомендуют писать на `chat.create()`
([руководство по переходу](https://github.com/ai-forever/gigachat/blob/efdc363a164b3a4def2f77af8c00bd4a73e4d971/MIGRATION_GUIDE.md)).

Оба метода принимают запрос в трёх видах: строкой (`client.chat.create("Привет")`), словарём или
типизированной моделью. В ноутбуке используем словари: так видна форма контракта. В прикладном коде
удобнее модели — их проверяет Pydantic и подсказывает редактор.

Версия API, версия SDK и название модели — три независимые настройки. Одна и та же модель
`GigaChat-3-Ultra` отвечает через оба контракта.

### 2.2. Запрос через v2

Создаём клиент без аргументов — ключ, scope и модель он прочитает из окружения. Отправляем
один вопрос. В v2 содержимое сообщения — список частей; текстовая часть — словарь с полем `text`.

In [ ]:
import gigachat

PROMPT = "Одним предложением: зачем приложению API языковой модели?"
sdk = gigachat.GigaChat()

payload_v2 = {"model": MODEL, "messages": [{"role": "user", "content": [{"text": PROMPT}]}]}
response_v2 = sdk.chat.create(payload_v2)

text_v2 = "".join(part.text or "" for message in response_v2.messages for part in message.content or [])
print("текст:", text_v2)
print("finish_reason:", response_v2.finish_reason)
print("usage:", response_v2.usage)
print("x_headers:", response_v2.x_headers)
show(response_v2, "Полный ответ v2")

Разберём, что пришло. Раскройте полный ответ и найдите поля:

- **`messages`** — список сообщений модели. Обычно одно, с `role: "assistant"`. Список нужен потому,
  что в одном ответе модель может вернуть несколько сообщений — например, вызов функции и текст.
- **`messages[].content[]`** — части сообщения. Текст лежит в поле `text` каждой части. Поэтому код
  выше не берёт готовую строку, а собирает текст из всех частей всех сообщений.
- **`finish_reason`** — почему модель остановилась. `stop` — закончила ответ сама; `length` —
  упёрлась в лимит длины; `function_call` — просит вызвать функцию. Полный список — в
  [справке v2](https://developers.sber.ru/docs/ru/gigachat/api/reference/rest/post-chat-v-2).
- **`model`** — фактическая модель с номером сборки, например `GigaChat-3-Ultra:32.9.23.6`.
- **`created_at`** — время ответа в секундах Unix.
- **`usage`** — расход токенов: `input_tokens` (вход), `output_tokens` (выход), `total_tokens`
  и `input_tokens_details.cached_tokens` — сколько токенов входа взято из кэша (раздел 4).

Поле `x_headers` в ответе сервера отсутствует. Его добавляет SDK из HTTP-заголовков ответа:
- **`x-request-id`** — идентификатор этого запроса. По нему запрос находят в журналах, например
  при обращении в поддержку.
- **`x-session-id`** — идентификатор сессии, то есть серии запросов. От него зависит кэш (раздел 4).

Оба идентификатора можно передать и в запросе — тогда сервер вернёт ваши значения. Если их не
передать, сервер сгенерирует новые.

### 2.3. Тот же вопрос через v1

В v1 содержимое сообщения — обычная строка, а текст ответа лежит по пути
`choices[0].message.content`.

In [ ]:
payload_v1 = {"model": MODEL, "messages": [{"role": "user", "content": PROMPT}]}
response_v1 = sdk.chat(payload_v1)

print("текст:", response_v1.choices[0].message.content)
print("finish_reason:", response_v1.choices[0].finish_reason)
print("usage:", response_v1.usage)
show(response_v1, "Полный ответ v1")

В ответе v1:

- **`choices`** — список вариантов ответа. Исторически API умели возвращать несколько вариантов
  на один запрос; сейчас вариант всегда один, но форма осталась.
- **`choices[0].message`** — сообщение модели: `role` и `content` — строка с текстом.
- **`choices[0].finish_reason`** — причина остановки, те же значения, что в v2.
- **`created`**, **`model`**, **`object`** — время, модель и метка типа объекта
  (`"chat.completions"`), унаследованная от формата OpenAI.
- **`usage`** — `prompt_tokens`, `completion_tokens`, `total_tokens` и `precached_prompt_tokens`.

Формулировки ответов v1 и v2 почти наверняка различаются, даже если смысл одинаков: модель
каждый раз генерирует текст заново.

| | v1 | v2 |
|---|---|---|
| Текст запроса | `content` — строка | `content` — список частей `[{"text": ...}]` |
| Текст ответа | `choices[0].message.content` | `messages[].content[].text` |
| Причина остановки | `choices[0].finish_reason` | `finish_reason` |
| Время | `created` | `created_at` |
| Вход и выход | `prompt_tokens`, `completion_tokens` | `input_tokens`, `output_tokens` |
| Токены из кэша | `precached_prompt_tokens` | `input_tokens_details.cached_tokens` |

Практическое следствие: при переходе с одного контракта на другой меняется и код, который
формирует запрос, и код, который читает ответ. Второй ломается чаще.

### 2.4. Две ловушки v2

**Параметры генерации лежат в `model_options`.** В v1 параметры вроде `max_tokens` стоят на
верхнем уровне тела. В v2 их нужно класть в объект `model_options`. Если оставить параметр на
верхнем уровне, сервер не вернёт ошибку и просто его проигнорирует. Проверим на `max_tokens`:

In [ ]:
question = [{"role": "user", "content": [{"text": "Объясни, что такое HTTP API."}]}]
for title, extra in [("max_tokens в model_options", {"model_options": {"max_tokens": 10}}),
                     ("max_tokens на верхнем уровне", {"max_tokens": 10})]:
    answer = sdk.chat.create({"model": MODEL, "messages": question, **extra})
    print(f"{title}: finish_reason={answer.finish_reason}, output_tokens={answer.usage.output_tokens}")

Во втором случае лимит не сработал: `finish_reason` равен `stop`, а ответ длиннее десяти токенов.
Код, механически перенесённый с v1, продолжит работать — но без лимита. Поэтому после изменения
запроса полезно проверять по ответу, что параметр действительно применился.

**Строка вместо списка частей.** Если отправить в v2 `content` строкой в сыром HTTP-запросе, сервер
ответит `400` с сообщением «Your request contains invalid JSON syntax», хотя JSON корректный:
неверна только форма поля. SDK эту ошибку предотвращает — строку он сам превращает в список
частей. Об этом стоит помнить, когда запросы пишутся без SDK.

## 3. Параметры генерации

### 3.1. Как модель выбирает следующий токен

Языковая модель пишет ответ по одному токену. На каждом шаге она оценивает вероятность каждого
возможного продолжения, а затем одно из продолжений выбирается. Например, после «Рыжий кот»
вероятности могут быть такими: «Мурзик» — 40 %, «Васька» — 25 %, «Пиксель» — 10 %, остальные
варианты делят оставшиеся 25 % (числа иллюстративные).

Выбор управляется двумя параметрами:

- **`temperature`** меняет форму распределения. Низкая температура заостряет его: лидер забирает
  почти всю вероятность, и выбор становится предсказуемее. Высокая — выравнивает: редкие варианты
  получают больше шансов, текст становится разнообразнее, а при слишком высоких значениях —
  бессвязным.
- **`top_p`** (nucleus sampling) обрезает хвост распределения: выбор идёт только среди самых
  вероятных вариантов, которые вместе набирают долю `top_p`. При `top_p = 0.5` в примере выше
  остались бы «Мурзик» и «Васька».

Обычно меняют один из двух параметров, а второй оставляют по умолчанию.

### 3.2. Параметры GigaChat

В v1 параметры задаются на верхнем уровне тела запроса, в v2 — внутри `model_options`.
Допустимые значения ниже взяты из ответов сервера на неверные значения (ошибка `422`).

| Параметр | Что делает | Допустимые значения |
|---|---|---|
| `temperature` | случайность выбора | больше 0; уже при 2 текст становится бессвязным |
| `top_p` | доля вероятности, среди которой идёт выбор | от 0 до 1 |
| `max_tokens` | предел длины ответа в токенах | больше 0; при достижении — `finish_reason: "length"` |
| `repetition_penalty` | штраф за повторы; 1 — без штрафа | больше 0 |
| `stream` | потоковая выдача ответа частями (раздел 5) | `true` / `false` |
| `update_interval` | как часто присылать части при потоковой выдаче | секунды |
| `response_format` | ответ по JSON Schema (структурированный вывод) | объект со схемой |

Параметра `seed`, который в некоторых API фиксирует случайность, у GigaChat нет.

Описание полей `model_options` — в [справке v2](https://developers.sber.ru/docs/ru/gigachat/api/reference/rest/post-chat-v-2).

### 3.3. Воспроизводимость

Можно ли, выставив низкую температуру, получать одинаковые ответы? Проверим: три одинаковых
запроса с `temperature = 0.01`. Вопрос выбран так, чтобы ответ был длинным: на коротких ответах
совпадения бывают часто и создают ложное впечатление.

In [ ]:
story = [{"role": "user", "content": [{"text": "Напиши короткую историю из трёх предложений про кота-программиста."}]}]
answers = []
for _ in range(3):
    answer = sdk.chat.create({"model": MODEL, "messages": story,
                              "model_options": {"temperature": 0.01, "max_tokens": 120}})
    answers.append(answer.messages[0].content[0].text)
print("разных ответов из трёх:", len(set(answers)))
for text in answers:
    print("  —", text[:90].replace("\n", " "), "…")

Скорее всего, ответы различаются. Проверки на моделях GigaChat-2 и GigaChat-3 показали: ни низкая
температура, ни `top_p = 0`, ни их сочетание не дают одинаковых длинных ответов. При низких
значениях разнообразие меньше — ответы начинаются с одного-двух вариантов, — но полной
повторяемости нет. Причина скрыта на стороне сервера.

Отсюда правило для разработки: тесты и проверки строятся на **структуре** ответа и **данных**
в нём — наличии вызова функции, значениях аргументов, числах из справочника. Совпадение текста
проверять бессмысленно.

Последний параметр этого раздела — `max_tokens`. Маленький лимит обрывает ответ на полуслове,
и `finish_reason` сообщает об этом:

In [ ]:
short = sdk.chat.create({"model": MODEL, "messages": question, "model_options": {"max_tokens": 15}})
print("текст:", short.messages[0].content[0].text)
print("finish_reason:", short.finish_reason)

Обрезанный ответ приложение должно распознавать по `finish_reason: "length"` и не считать
завершённым. Особенно это важно для вызова функций: обрезанные аргументы исполнять нельзя.

## 4. Кэш и стоимость

### 4.1. Что оплачивается

Токены GigaChat оплачиваются по тарифу модели или списываются с пакета токенов. Оплачиваются
входные и выходные токены. Токены входа, которые сервер взял из **кэша**, не тарифицируются —
это прямо сказано в [документации](https://developers.sber.ru/docs/ru/gigachat/guides/counting-tokens)
и подтверждается по балансу: с пакета списывается ровно `входные + выходные` без закэшированных.

Кэш работает так: если начало запроса совпадает с началом недавнего запроса **той же сессии**,
сервер не обрабатывает его заново. Сессию задаёт заголовок `X-Session-ID`. Без него сервер
присваивает каждому запросу новую сессию, и кэш почти никогда не срабатывает.

В SDK заголовок задаётся контекстной переменной `gigachat.session_id_cvar`: все запросы в этом
контексте получат одинаковый `X-Session-ID`.

### 4.2. Опыт

Отправим дважды длинный одинаковый контекст — регламент на несколько тысяч символов — сначала
без сессии, затем с одной сессией, и посмотрим на `usage`.

In [ ]:
import uuid

long_context = "Регламент учебной части: " + "аудитории бронируются заранее через учебный отдел. " * 120
cache_messages = [{"role": "system", "content": [{"text": long_context}]},
                  {"role": "user", "content": [{"text": "Ответь одним словом: о чём текст?"}]}]


def cached_usage():
    answer = sdk.chat.create({"model": MODEL, "messages": cache_messages, "model_options": {"max_tokens": 5}})
    details = answer.usage.input_tokens_details
    return answer.usage.input_tokens, details.cached_tokens if details else 0


print("без X-Session-ID:", [cached_usage() for _ in range(2)])
session = gigachat.session_id_cvar.set(str(uuid.uuid4()))
print("с одним X-Session-ID:", [cached_usage() for _ in range(2)])
gigachat.session_id_cvar.reset(session)

Пары чисел — это `(input_tokens, cached_tokens)`. Без сессии кэш срабатывает редко и непредсказуемо:
обычно оба запроса обработаны полностью, но бывает, что часть взята из кэша. С одной сессией второй запрос почти целиком взят из кэша: `input_tokens` — несколько десятков
токенов нового хвоста, остальное — в `cached_tokens`. Обратите внимание: `input_tokens` в GigaChat
**не включает** закэшированные токены. Полный размер входа — сумма двух полей.

Где это важно:
- **агенты**, у которых история растёт с каждым шагом и начало запроса повторяется;
- **длинные системные инструкции** и большие документы в контексте;
- **подсчёт стоимости**: в домашнем задании расход берётся из `usage`, и без `X-Session-ID`
  закэшированных токенов почти не будет.

## 5. Другие методы API

Кроме генерации ответа, у API есть служебные методы. Три из них ниже не вызывают генерацию
и почти ничего не стоят. Полный список методов — в [справке GigaChat API](https://developers.sber.ru/docs/ru/gigachat/api/reference/rest/gigachat-api)
и в [README SDK](https://github.com/ai-forever/gigachat).

**Список моделей** — какие модели доступны аккаунту. Отсюда берётся значение для `GIGACHAT_MODEL`.

In [ ]:
print([model.id_ for model in sdk.get_models().data])

Помимо моделей генерации в списке есть модели эмбеддингов — они превращают текст в векторы для
поиска по смыслу. Их разберём в блоке про RAG.

**Подсчёт токенов** — сколько токенов займёт текст, до отправки запроса. Полезно, чтобы проверить,
поместится ли документ в контекст модели, и оценить стоимость.

In [ ]:
counted = sdk.tokens_count([PROMPT], model=MODEL)[0]
print(f"токенов: {counted.tokens}, символов: {counted.characters}")
print("prompt_tokens того же вопроса в запросе v1:", response_v1.usage.prompt_tokens)

`tokens_count` считает токены самого текста. В запросе к модели входных токенов больше: к тексту
добавляется служебная разметка диалога — роли и границы сообщений.

**Баланс** — остаток токенов по пакетам. У аккаунта физического лица пакеты выделены по семействам
моделей.

In [ ]:
for item in sdk.get_balance().balance:
    print(f"{item.usage}: {item.value:,.0f}".replace(",", " "))

**Потоковая выдача.** Обычно ответ приходит целиком, когда генерация закончена. С `stream: true`
сервер присылает ответ частями по мере генерации — так чат-интерфейсы показывают текст
«печатающимся». В SDK для v1 это метод `client.stream(...)`, для v2 — `client.chat.stream(...)`;
оба возвращают итератор частей.

In [ ]:
import time

started = time.perf_counter()
stream_payload = {"model": MODEL, "max_tokens": 400,
                  "messages": [{"role": "user", "content": "Подробно, в шести абзацах, объясни, как работает HTTP."}]}
for number, chunk in enumerate(sdk.stream(stream_payload), start=1):
    delta = chunk.choices[0].delta.content or ""
    print(f"[{time.perf_counter() - started:4.1f} с] часть {number}: {delta[:50]!r}")

Каждая часть содержит `choices[0].delta.content` — приращение текста; приложение склеивает части
само. Смысл потоковой выдачи в том, чтобы первая часть пришла раньше, чем закончится вся генерация.

Посмотрите на метки времени: первая часть приходит меньше чем через секунду, последняя — через
несколько секунд, когда генерация закончится. При подготовке занятия длинный ответ `GigaChat-3-Ultra`
приходил с 0,6 до 7,4 секунды. Поэтому для опыта взят длинный ответ: на ответе из пары предложений
вся генерация занимает около секунды, и разницу трудно заметить.

Вызов функции (раздел 7) в потоке не дробится: имя и аргументы приходят одной частью, целиком,
вместе с `finish_reason: "function_call"`.

В остальных разделах используем обычные запросы — так проще читать ответы.

## 6. Зачем модели инструмент

Вернёмся к главному вопросу занятия. Аудитория A-101 существует только в нашем учебном справочнике
`room_tool.py`: данные вымышлены, и модель их видеть не может. Спросим модель напрямую, без доступа
к справочнику. До запуска подумайте: что она ответит?

In [ ]:
QUESTION = "Поместятся ли 28 студентов в аудитории A-101?"

def user_v2(text):
    return {"role": "user", "content": [{"text": text}]}


baseline = sdk.chat.create({"model": MODEL, "messages": [user_v2(QUESTION)]})
print(baseline.messages[0].content[0].text)
print("\nfinish_reason:", baseline.finish_reason)

Посмотрите, что пришло. Обычно модель честно пишет, что не знает вместимость конкретной аудитории,
и советует уточнить в учебной части. Иногда она называет типичную вместимость — например, «от 25
до 40 человек». Такое число правдоподобно, но это догадка: заданную нами вместимость модель
проверить не может, потому что справочник ей не передали.

Если приложение покажет такую догадку пользователю как ответ, пользователь получит ошибку.
Задача приложения — дать модели доступ к данным в момент ответа. Механизм, которым это делается,
называется **вызовом инструмента** (tool calling, function calling).

## 7. Вызов функции

### 7.1. Схема обмена

```text
Приложение                                    Модель
    │── вопрос + описание функции ───────────────►│
    │◄── имя функции + аргументы ─────────────────│   finish_reason: function_call
    │                                             │
    │  проверяет вызов и исполняет функцию        │
    │  get_room_info("A-101") → 24 места          │
    │                                             │
    │── история + результат функции ─────────────►│
    │◄── ответ на исходный вопрос ────────────────│   finish_reason: stop
```

Четыре опоры, к которым будем возвращаться во всех форматах:

1. **Описание функции.** Модель видит только имя, описание и схему аргументов. Код функции
   остаётся в приложении.
2. **Вызов.** Модель просит исполнить функцию: присылает её имя и аргументы.
3. **Исполнение.** Функцию исполняет приложение — после проверки, что вызов допустим.
4. **Результат.** Приложение делает второй запрос: передаёт всю историю и результат функции.
   Модель отвечает уже с учётом данных.

Первый запрос на этом завершён. Во второй запрос приложение кладёт всю нужную историю: в этом
режиме сервер её не хранит.

### 7.2. Справочник и проверка вызова

`get_room_info` — обычная функция Python: читает словарь и возвращает вместимость и наличие
проектора. `execute_tool` — точка входа для вызовов от модели. Она проверяет, что имя функции
известно, аргумент ровно один — `room`, и аудитория есть в справочнике. Модель может ошибиться
в имени или аргументах, поэтому приложение никогда не исполняет вызов без проверки. Наш
инструмент только читает данные; для функций, которые что-то меняют, понадобятся ещё проверки
прав и подтверждение пользователя.

Следующая ячейка задаёт исходное состояние справочника. К ней можно вернуться после упражнения,
чтобы восстановить данные.

In [ ]:
# Сброс учебного справочника
from room_tool import ROOMS, execute_tool, get_room_info

ROOMS["A-101"]["capacity"] = 24
ROOMS["B-202"]["capacity"] = 40
print(get_room_info("A-101"))
print(get_room_info("B-202"))

### 7.3. Описание функции

Функцию описывают тремя полями: `name`, `description` и `parameters`. Поле `parameters` — это
[JSON Schema](https://json-schema.org/): стандартное описание структуры данных. Здесь это объект
с одним обязательным строковым полем `room`. Описания функции и поля модель читает, чтобы понять,
когда вызывать функцию и что передавать, поэтому их пишут так же внимательно, как промпт.

Этот словарь — общий для всех форматов занятия: меняться будет только обёртка вокруг него.

В v2 пользовательские функции передаются в списке инструментов `tools` элементом
`{"functions": {"specifications": [...]}}`.

In [ ]:
FUNCTION = {
    "name": "get_room_info",
    "description": get_room_info.__doc__,
    "parameters": {
        "type": "object",
        "properties": {"room": {"type": "string", "description": "Код аудитории, например A-101"}},
        "required": ["room"],
    },
}
GIGACHAT_TOOLS = [{"functions": {"specifications": [FUNCTION]}}]
show(GIGACHAT_TOOLS, "Инструменты в формате v2")

### 7.4. Модель просит вызвать функцию

In [ ]:
first = sdk.chat.create({"model": MODEL, "messages": [user_v2(QUESTION)], "tools": GIGACHAT_TOOLS})
call_message = first.messages[0]
call = next(part.function_call for part in call_message.content if part.function_call)

print("finish_reason:", first.finish_reason)
print("вызов:", call.name, call.arguments)
print("tool_state_id:", call_message.tools_state_id)
show(first, "Полный ответ с вызовом")

Что пришло:

- `finish_reason: "function_call"` — модель остановилась, чтобы приложение выполнило функцию.
- Вызов лежит **частью** `content` сообщения ассистента: `{"function_call": {"name", "arguments"}}`.
  Текста в сообщении нет.
- `arguments` — уже объект (словарь), разбирать JSON не нужно.
- `tool_state_id` — идентификатор состояния вызова на стороне GigaChat.

### 7.5. Исполняем функцию и возвращаем результат

Приложение проверяет вызов и исполняет функцию через `execute_tool`. Затем собирает историю
для второго запроса:

1. исходный вопрос пользователя;
2. сообщение модели с вызовом — целиком, как оно пришло;
3. результат: сообщение с ролью `tool`, в котором часть `function_result` с именем функции
   и результатом-объектом.

In [ ]:
result = execute_tool(call.name, call.arguments)
history = [
    user_v2(QUESTION),
    call_message.model_dump(exclude_none=True, by_alias=True),
    {"role": "tool", "content": [{"function_result": {"name": call.name, "result": result}}]},
]
show(history, "История для второго запроса")

final = sdk.chat.create({"model": MODEL, "messages": history, "tools": GIGACHAT_TOOLS})
print("finish_reason:", final.finish_reason)
print("ответ:", final.messages[0].content[0].text)

Теперь модель отвечает по нашим данным: в A-101 двадцать четыре места, двадцать восемь студентов
не поместятся. Сравните с ответом из раздела 6: изменилось только то, что приложение передало
модели факт из справочника.

Результат функции GigaChat требует в виде JSON: объект или JSON-строка. Произвольный текст
сервер отклоняет с ошибкой `422`.

### 7.6. Режимы вызова

Поле `tool_config` управляет тем, вызывает ли модель функции:

| `tool_config` | Поведение |
|---|---|
| не задан или `{"mode": "auto"}` | модель решает сама — это режим по умолчанию |
| `{"mode": "none"}` | функции не вызываются |
| `{"mode": "forced", "function_name": "get_room_info"}` | вызов этой функции обязателен |

Неизвестное значение `mode` сервер принимает без ошибки и работает как в режиме `auto`, поэтому
опечатка в названии режима останется незамеченной. Проверяйте по ответу, что режим сработал.

Принудительный вызов удобен для демонстраций и тестов. Но модель выполнит его даже там, где он
не имеет смысла, — и придумает аргументы. Проверим на приветствии:

In [ ]:
forced = sdk.chat.create({"model": MODEL, "messages": [user_v2("Привет!")], "tools": GIGACHAT_TOOLS,
                          "tool_config": {"mode": "forced", "function_name": "get_room_info"}})
forced_call = next(part.function_call for part in forced.messages[0].content if part.function_call)
print("аргументы принудительного вызова:", forced_call.arguments)
try:
    print("execute_tool исполнил вызов:", execute_tool(forced_call.name, forced_call.arguments))
except ValueError as error:
    print("execute_tool отклонил вызов:", error)

Посмотрите на аргумент. Пользователь про аудиторию не спрашивал, и аргумент модель придумала.
При подготовке занятия встречались два исхода: пустая строка — её `execute_tool` отклоняет, — и
правдоподобное `"A-101"` — такой вызов проверку проходит и исполняется. Проверка ловит вызовы
неверной формы и несуществующие аудитории; придуманный, но правдоподобный аргумент она не распознает.
Поэтому принудительный режим оставляют для тестов и демонстраций.

### 7.7. Несколько вызовов: цикл

Спросим про две аудитории. Модель вызывает функцию по одному разу за ответ, поэтому приложение
повторяет шаги, пока модель просит вызовы: получить вызов → исполнить → вернуть результат →
снова спросить модель. Этот цикл — ядро любого агента. На следующем занятии его возьмёт на себя
агентный фреймворк.

In [ ]:
history = [user_v2("Сравни вместимость аудиторий A-101 и B-202.")]
for step in range(1, 6):
    answer = sdk.chat.create({"model": MODEL, "messages": history, "tools": GIGACHAT_TOOLS})
    message = answer.messages[0]
    calls = [part.function_call for part in message.content if part.function_call]
    if answer.finish_reason != "function_call":
        print(f"шаг {step}: ответ — {message.content[0].text}")
        break
    results = []
    for call in calls:
        result = execute_tool(call.name, call.arguments)
        print(f"шаг {step}: {call.name}({call.arguments}) → {result}")
        results.append({"function_result": {"name": call.name, "result": result}})
    history += [message.model_dump(exclude_none=True, by_alias=True), {"role": "tool", "content": results}]

Код готов к нескольким вызовам в одном ответе — формат v2 это допускает, — но модель, как правило,
просит их по одному.

## 8. Встроенный инструмент

Кроме своих функций, в `tools` можно включить инструменты, которые исполняет **сам сервер**
GigaChat: `web_search` (поиск в интернете), `code_interpreter`, `image_generate`,
`url_content_extraction`, `model_3d_generate`. Приложению ничего исполнять не нужно: сервер сам
выполняет поиск, подставляет найденное в контекст модели и возвращает готовый ответ.

В SDK встроенный инструмент можно указать строкой: `"web_search"`. SDK превращает её в объект
`{"web_search": {}}` — именно такую форму ждёт сервер. Подробнее — в документации о
[встроенных функциях](https://developers.sber.ru/docs/ru/gigachat/guides/functions/calling-builtin-functions).

In [ ]:
searched = sdk.chat.create({"model": MODEL, "tools": ["web_search"],
                            "messages": [user_v2("Какой курс доллара ЦБ РФ на сегодня? Коротко, с источником.")]})
part = searched.messages[0].content[0]
print("ответ:", part.text)
print("\nисточники:")
for number, source in (part.inline_data.sources or {}).items():
    print(f"  [{number}] {source.title} — {source.url}")
print("\nusage:", searched.usage)

Что видно:

- В тексте — пометки вида `[sources=[2]]`: номер источника из `inline_data.sources`.
- Сообщений с вызовом поиска и его результатом в ответе нет — всё произошло на сервере. При потоковой
  выдаче видны только события о ходе выполнения инструмента.
- `input_tokens` намного больше длины вопроса: найденные страницы подставлены в контекст модели
  и считаются входными токенами.

Сравните с нашей функцией. Там весь обмен в руках приложения: вызов, аргументы, результат, история.
Здесь приложение получает готовый ответ, а на чём он основан, можно проверить только по списку
источников.

## 9. GPT2GIGA: GigaChat в форматах OpenAI и Anthropic

### 9.1. Зачем нужен прокси

Многие библиотеки и инструменты — от редакторов кода до агентных фреймворков — умеют работать
только с форматами OpenAI или Anthropic. Эти форматы стали негласным стандартом, и формулировка
«совместимо с OpenAI» чаще всего означает именно их.

[GPT2GIGA](https://github.com/ai-forever/gpt2giga) — открытый прокси от команды GigaChat. Он
принимает запросы в форматах OpenAI (Chat Completions, Responses) и Anthropic (Messages), переводит
их в запросы к GigaChat, а ответ GigaChat переводит обратно. Клиентская библиотека думает, что
разговаривает со своим сервисом; отвечает же GigaChat. Так к GigaChat подключают готовые
инструменты, не переписывая их.

На занятии прокси нужен для другого: сравнить три популярных формата на одной модели и одном примере.
Возможности каждого формата при этом ограничены тем, что реализовано в прокси и что умеет GigaChat.

### 9.2. Две точки наблюдения

```text
Ноутбук (SDK OpenAI / Anthropic)
        │  A: запрос в формате Chat Completions / Responses / Messages
        ▼
GPT2GIGA (127.0.0.1:8090)
        │  B: запрос в формате GigaChat
        ▼
GigaChat API
```

- **Участок А** — между библиотекой и прокси. Его покажем прямо в ноутбуке: перехватим запрос,
  который библиотека подготовила к отправке.
- **Участок Б** — между прокси и GigaChat. Его показывает [mitmproxy](https://mitmproxy.org/) —
  программа для перехвата и просмотра HTTP(S)-трафика с веб-интерфейсом. Этот шаг необязательный.

### 9.3. Запуск

Прокси — отдельный долгоживущий процесс со своими логами, поэтому его удобно держать в отдельном
терминале. Во втором терминале, в папке `llm_api_to_langchain`:

```bash
uv run tools/run_proxy.py
```

Скрипт запускает GPT2GIGA 0.3.0 через `uvx` с закреплёнными версиями зависимостей и настройками
из `.env`. Прокси слушает `127.0.0.1:8090` — он доступен только с вашего компьютера — и работает в
учебном режиме без собственного ключа доступа. Настоящий ключ GigaChat знает только прокси.
Остановка — `Ctrl+C` в этом терминале.

Для участка Б — **до** запуска прокси, в третьем терминале:

```bash
uv run tools/start_mitmproxy.py
```

Он запускает mitmproxy с веб-интерфейсом на `http://127.0.0.1:8081` (ссылку для входа скрипт
печатает в терминал). `run_proxy.py` сам находит mitmproxy и направляет через него запросы к
GigaChat; без mitmproxy прокси работает напрямую. Запрос токена GigaChat mitmproxy пропускает
без расшифровки, а заголовок `Authorization` в интерфейсе скрыт.

### 9.4. Как увидеть запрос на участке А

Библиотеки OpenAI и Anthropic отправляют запросы через HTTP-клиент `httpx2`. Ему можно передать
**hook** — функцию, которую клиент вызывает перед отправкой каждого запроса. Наш hook печатает
метод и путь, а под ними — раскрывающийся блок с телом запроса: ровно те байты, которые библиотека
подготовила к отправке. Заголовки не печатаются: в них ключ.

In [ ]:
import httpx2


def print_http_request(request):
    print(f"HTTP → {request.method} {request.url.path}")
    body = request.content.decode("utf-8")
    display(HTML(f"<details><summary>Тело HTTP-запроса</summary><pre>{escape(body)}</pre></details>"))

Следующая ячейка создаёт клиентов OpenAI и Anthropic. Сначала она проверяет, что адреса из `.env`
ведут на локальный прокси и прокси отвечает. Затем создаёт клиентов:

- `base_url` — адрес прокси. Для OpenAI — с путём `/v1/`. Для Anthropic — без `/v1`: библиотека
  сама добавляет к адресу `/v1/messages`, и лишний `/v1` дал бы адрес `/v1/v1/messages` и ошибку `404`.
- `api_key="unused"` — библиотекам нужна непустая строка, а прокси в учебном режиме ключ не проверяет.
- `max_retries=0` — библиотека не повторяет запрос при ошибке: каждую ошибку видно сразу.
- `http_client` — HTTP-клиент с нашим hook.

In [ ]:
import socket
from urllib.parse import urlsplit

import httpx
from anthropic import Anthropic
from openai import OpenAI

for key in ("OPENAI_BASE_URL", "ANTHROPIC_BASE_URL"):
    url = urlsplit(os.environ[key])
    assert url.scheme == "http" and url.hostname == "127.0.0.1", "Для занятия используем локальный прокси"
assert urlsplit(os.environ["OPENAI_BASE_URL"]).path == "/v1/"
assert urlsplit(os.environ["ANTHROPIC_BASE_URL"]).path in {"", "/"}
port = int(os.getenv("GPT2GIGA_PORT", "8090"))
httpx.get(f"http://127.0.0.1:{port}/health", timeout=5).raise_for_status()
print("Прокси отвечает. Доступ к модели проверит первый запрос.")

with socket.socket() as probe:
    probe.settimeout(0.5)
    mitm_running = probe.connect_ex(("127.0.0.1", 8080)) == 0
print("mitmproxy:", "запущен — участок Б в http://127.0.0.1:8081" if mitm_running else "не запущен (необязательно)")

openai_client = OpenAI(base_url=os.environ["OPENAI_BASE_URL"], api_key="unused", timeout=60, max_retries=0,
                       http_client=httpx2.Client(event_hooks={"request": [print_http_request]}))
anthropic_client = Anthropic(base_url=os.environ["ANTHROPIC_BASE_URL"], api_key="unused", timeout=60, max_retries=0,
                             http_client=httpx2.Client(event_hooks={"request": [print_http_request]}))

Запрос `/health` показывает только, что процесс прокси отвечает. Работает ли ключ GigaChat и есть ли
доступ к модели, станет ясно на первом настоящем запросе.

## 10. OpenAI Chat Completions

Chat Completions — самый распространённый формат обмена с языковыми моделями. Его придумала OpenAI;
GigaChat v1 устроен по его раннему образцу, поэтому многое будет знакомо.
Документация: [Chat Completions](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create),
[вызов функций](https://developers.openai.com/api/docs/guides/function-calling).

### 10.1. Описание функции

Функции передаются в поле `tools`. Каждый инструмент — обёртка `{"type": "function", "function": ...}`,
внутри — тот же словарь `FUNCTION`, что для GigaChat.

In [ ]:
SYSTEM = "Для сведений об аудитории используй инструмент. Отвечай кратко по-русски."
CHAT_TOOLS = [{"type": "function", "function": FUNCTION}]
show(CHAT_TOOLS, "Инструменты в формате Chat Completions")

### 10.2. Обмен

Системная инструкция в этом формате — первое сообщение списка `messages` с ролью `system`.
Идём по четырём опорам из раздела 7:

1. **Вызов.** В ответе `message.tool_calls` — список вызовов; `finish_reason: "tool_calls"`.
   У каждого вызова есть `id`, а аргументы `function.arguments` приходят **JSON-строкой**:
   так их сериализует этот формат, поэтому приложение разбирает их через `json.loads`.
2. **Исполнение** — та же `execute_tool`.
3. **Результат** — сообщение с ролью `tool`, полем `tool_call_id` со ссылкой на вызов и строкой
   в `content`. Сообщение модели с `tool_calls` кладётся в историю целиком.
4. **Второй запрос** — вся история.

Следите за строками `HTTP →` и раскрывайте тело запросов: во втором запросе видна вся история.

In [ ]:
chat_messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": QUESTION}]
first_chat = openai_client.chat.completions.create(model=MODEL, messages=chat_messages, tools=CHAT_TOOLS)
assistant_message = first_chat.choices[0].message
print("finish_reason:", first_chat.choices[0].finish_reason)
print("tool_calls:", [call.model_dump() for call in assistant_message.tool_calls or []])
assert assistant_message.tool_calls, "Нет вызова функции: остановитесь и разберите ответ модели"

chat_messages.append(assistant_message.model_dump(exclude_none=True))
for tool_call in assistant_message.tool_calls:
    result = execute_tool(tool_call.function.name, json.loads(tool_call.function.arguments))
    chat_messages.append({"role": "tool", "tool_call_id": tool_call.id,
                          "content": json.dumps(result, ensure_ascii=False)})

final_chat = openai_client.chat.completions.create(model=MODEL, messages=chat_messages, tools=CHAT_TOOLS)
print("ответ:", final_chat.choices[0].message.content)
show(final_chat, "Полный финальный ответ")

Что стоит заметить:

- Ответ похож на GigaChat v1: `choices[0].message`, `finish_reason`, `usage`. Добавились поля формата
  OpenAI — `id`, `system_fingerprint`, `usage.prompt_tokens_details.cached_tokens`. Их у GigaChat нет,
  поэтому `id` и `system_fingerprint` генерирует прокси.
- В полном ответе есть и лишние для OpenAI поля — `functions_state_id`, `data_for_context` и другие.
  Прокси переносит их из ответа GigaChat. Клиенту OpenAI они не мешают, но показывают, что формат
  воспроизведён приблизительно.
- Идентификатор вызова в `tool_calls[].id` — это `functions_state_id` из ответа GigaChat.

Режим вызова задаёт `tool_choice`: `"auto"` (по умолчанию), `"none"`, `"required"` или
`{"type": "function", "function": {"name": ...}}`. Режим `"required"` — «вызови какую-нибудь
функцию» — прокси не передаёт в GigaChat: у GigaChat такого режима нет. Запрос обрабатывается так,
будто стоит `"auto"`.

## 11. OpenAI Responses

Responses — более новый формат OpenAI. В нём вход `input` и выход `output` — списки
**типизированных элементов**: сообщение, вызов функции, результат функции и так далее.
Документация: [переход от Chat Completions к Responses](https://developers.openai.com/api/docs/guides/migrate-to-responses).

| Опора | Как в Responses |
|---|---|
| Инструкции | отдельное поле `instructions` |
| Описание функции | плоское: `{"type": "function", "name", "description", "parameters"}` |
| Вызов | элемент `output` с `type: "function_call"`; `arguments` — JSON-строка; `call_id` |
| Результат | элемент `{"type": "function_call_output", "call_id", "output"}` во входе |

У элемента вызова два идентификатора: `id` — идентификатор самого элемента, `call_id` — связь
вызова с результатом. Результат ссылается на `call_id`.

In [ ]:
RESPONSES_TOOLS = [{"type": "function", **FUNCTION}]
responses_input = [{"role": "user", "content": QUESTION}]
first_response = openai_client.responses.create(model=MODEL, instructions=SYSTEM, input=responses_input,
                                                tools=RESPONSES_TOOLS, store=False)
print("элементы output:", [item.type for item in first_response.output])
responses_calls = [item for item in first_response.output if item.type == "function_call"]
print("вызовы:", [{"name": c.name, "arguments": c.arguments, "call_id": c.call_id} for c in responses_calls])
assert responses_calls, "Нет вызова функции: остановитесь и разберите ответ модели"

responses_input.extend(item.model_dump(exclude_none=True) for item in first_response.output)
for item in responses_calls:
    result = execute_tool(item.name, json.loads(item.arguments))
    responses_input.append({"type": "function_call_output", "call_id": item.call_id,
                            "output": json.dumps(result, ensure_ascii=False)})

final_response = openai_client.responses.create(model=MODEL, instructions=SYSTEM, input=responses_input,
                                                tools=RESPONSES_TOOLS, store=False)
print("ответ:", final_response.output_text)
show(final_response, "Полный финальный ответ")

Что стоит заметить:

- В `input` второго запроса кладём **все** элементы `output` первого ответа, включая вызов: рядом
  с вызовом там могут быть другие элементы, и модель должна видеть свой ответ целиком.
- `output_text` — удобное свойство библиотеки: оно собирает текст из всех элементов `message`.
- Ответ Responses возвращает эхом настройки запроса: `tools`, `tool_choice`, `temperature`, `top_p`.
  Значения `temperature` и `top_p`, которые мы не передавали, подставил прокси — о настройках
  GigaChat они ничего не говорят.
- `store=False` означает: продолжаем диалог явной передачей истории во `input`. В оригинальном
  Responses есть и другой способ — `store: true` и `previous_response_id`, когда историю хранит
  сервис. Через GPT2GIGA 0.3.0 этот способ не работает: параметры принимаются без ошибки и
  игнорируются, модель предыдущий ответ не видит.

## 12. Anthropic Messages

Messages — формат API Anthropic. Ответ модели — список **блоков** `content`: текст, вызов инструмента.
Отвечает по-прежнему GigaChat: модели Claude на занятии не вызываются.
Документация: [обработка вызовов инструментов](https://platform.claude.com/docs/en/agents-and-tools/tool-use/handle-tool-calls).

| Опора | Как в Messages |
|---|---|
| Инструкции | отдельное поле `system` |
| Описание функции | `name`, `description` и `input_schema` — та же JSON Schema под другим именем |
| Вызов | блок `{"type": "tool_use", "id", "name", "input"}`; `input` — объект |
| Результат | блок `{"type": "tool_result", "tool_use_id", "content"}` в сообщении с ролью **`user`** |

Результат приходит от «пользователя», потому что в этом формате диалог чередует только две стороны —
пользователя и модель; всё, что модель получает извне, идёт с ролью `user`. Параметр `max_tokens`
в API Anthropic обязателен, и библиотека требует его ещё до отправки.

In [ ]:
ANTHROPIC_TOOLS = [{"name": FUNCTION["name"], "description": FUNCTION["description"],
                    "input_schema": FUNCTION["parameters"]}]
anthropic_messages = [{"role": "user", "content": QUESTION}]
first_message = anthropic_client.messages.create(model=MODEL, system=SYSTEM, max_tokens=512,
                                                 messages=anthropic_messages, tools=ANTHROPIC_TOOLS)
print("stop_reason:", first_message.stop_reason)
tool_uses = [block for block in first_message.content if block.type == "tool_use"]
print("tool_use:", [{"name": b.name, "input": b.input, "id": b.id} for b in tool_uses])
assert tool_uses, "Нет вызова функции: остановитесь и разберите ответ модели"

anthropic_messages.append({"role": "assistant",
                           "content": [block.model_dump(exclude_none=True) for block in first_message.content]})
anthropic_messages.append({"role": "user", "content": [
    {"type": "tool_result", "tool_use_id": block.id,
     "content": json.dumps(execute_tool(block.name, block.input), ensure_ascii=False)}
    for block in tool_uses]})

final_message = anthropic_client.messages.create(model=MODEL, system=SYSTEM, max_tokens=512,
                                                 messages=anthropic_messages, tools=ANTHROPIC_TOOLS)
print("stop_reason:", final_message.stop_reason)
print("ответ:", "".join(block.text for block in final_message.content if block.type == "text"))
show(final_message, "Полный финальный ответ")

Что стоит заметить:

- `stop_reason` — аналог `finish_reason`: `tool_use` (просит вызов), `end_turn` (закончил ход),
  `max_tokens` (ответ обрезан — такие аргументы исполнять нельзя).
- Ответ короче, чем в форматах OpenAI: нет эха настроек и идентификаторов GigaChat.
- В `usage` только `input_tokens` и `output_tokens`. Сведения о закэшированных токенах прокси
  в этом формате не передаёт, поэтому полный размер входа по Messages не восстановить.

### 12.1. Три формата рядом

| | Chat Completions | Responses | Messages |
|---|---|---|---|
| Инструкции | сообщение `role: "system"` | `instructions` | `system` |
| Текст ответа | `choices[0].message.content` | элемент `message` → `output_text` | блоки `text` в `content[]` |
| Схема аргументов | `tools[].function.parameters` | `tools[].parameters` | `tools[].input_schema` |
| Вызов | `message.tool_calls[]` | элемент `function_call` | блок `tool_use` |
| Аргументы | JSON-строка | JSON-строка | объект |
| Результат | сообщение `role: "tool"` | элемент `function_call_output` | блок `tool_result` в сообщении `user` |
| Связь с вызовом | `tool_call_id` | `call_id` | `tool_use_id` |
| Причина остановки | `finish_reason` | тип элементов `output` | `stop_reason` |

Функция `get_room_info` при смене формата не менялась: она получает код аудитории и возвращает
данные. Меняется только упаковка — как описать функцию, как принять вызов и как вернуть результат.

## 13. Что прокси отправляет в GigaChat

Если mitmproxy запущен, откройте его интерфейс и найдите запросы к
`api.giga.chat/v1/chat/completions`, соответствующие обменам из разделов 10, 11 и 12. Если нет — ниже
показан перехват, сделанный при подготовке занятия.

Все три формата превращаются в **одинаковый запрос GigaChat v1**. Второй запрос обмена — с
результатом функции — на участке Б выглядит так:

```json
{
  "model": "GigaChat-3-Ultra",
  "messages": [
    {"role": "user", "content": "Поместятся ли 28 студентов в аудитории A-101?"},
    {"role": "assistant", "content": "",
     "function_call": {"name": "get_room_info", "arguments": {"room": "A-101"}}},
    {"role": "function", "name": "get_room_info",
     "content": "{\"room\": \"A-101\", \"capacity\": 24, \"projector\": true}"}
  ],
  "functions": [{"name": "get_room_info", "description": "…", "parameters": {"…": "…"}}]
}
```

Три наблюдения:

1. **Форматы сходятся в один.** `tools`, `tool_calls`, `function_call`, `tool_use` превращаются в
   `functions` и `function_call` GigaChat v1, а результат — в сообщение с ролью `function`.
2. **Идентификатор вызова до GigaChat не доходит.** В запросе нет ни `tool_call_id`, ни `call_id`,
   ни `tool_use_id`. GigaChat v1 связывает результат с вызовом по имени функции. Идентификаторы,
   которые мы тщательно передавали, нужны клиентскому формату и прокси: по ним прокси собирает
   историю. Модель их не видит.
3. **Строку прокси оборачивает в JSON.** Если вернуть результат инструмента простым текстом, прокси
   передаст в GigaChat `{"result": "…"}`: GigaChat принимает результат функции только в виде JSON.

Отсюда и главный смысл прокси: он переводит **форму** запросов и ответов. Модель, её возможности
и ограничения остаются прежними — это GigaChat.

Как настроить и читать перехват — в [HTTP_OBSERVATION.md](HTTP_OBSERVATION.md).

## 14. LangChain: общие объекты вместо форматов

### 14.1. Что даёт LangChain

Мы трижды упаковали один и тот же обмен в разные форматы. Приложение, которое работает с несколькими
провайдерами, пришлось бы писать под каждый формат отдельно. LangChain предлагает общий слой:
типы сообщений и интерфейс модели, одинаковые для всех провайдеров.

| Объект | Назначение |
|---|---|
| `SystemMessage` | инструкции |
| `HumanMessage` | сообщение пользователя |
| `AIMessage` | ответ модели, включая `tool_calls` — список вызовов |
| `ToolMessage` | результат функции со ссылкой на вызов |

Перевод этих объектов в запрос к конкретному API делает **пакет интеграции** провайдера. Для GigaChat
это [`langchain-gigachat`](https://github.com/ai-forever/langchain-gigachat), класс
`langchain_gigachat.GigaChat`. В версии 0.5.1 он обращается к GigaChat через контракт **v1**
(метод SDK `chat()`), поэтому возможности, которые есть только в v2, через него недоступны.

`bind_tools` передаёт модели описания инструментов. Исполнять функцию и добавлять результат в историю
по-прежнему будем сами — на следующем занятии это возьмёт на себя агентный цикл.

In [ ]:
import langchain_gigachat
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

room_tool = tool(get_room_info)
llm = langchain_gigachat.GigaChat()
lc_messages = [SystemMessage(content=SYSTEM), HumanMessage(content=QUESTION)]
ai_message = llm.bind_tools([room_tool]).invoke(lc_messages)
print("tool_calls:", ai_message.tool_calls)
print("usage_metadata:", ai_message.usage_metadata)
show(ai_message, "Полный AIMessage")
assert ai_message.tool_calls, "Нет вызова функции: остановитесь и разберите ответ модели"

lc_messages.append(ai_message)
for lc_call in ai_message.tool_calls:
    result = execute_tool(lc_call["name"], lc_call["args"])
    lc_messages.append(ToolMessage(content=json.dumps(result, ensure_ascii=False),
                                   tool_call_id=lc_call["id"], name=lc_call["name"]))
final_lc = llm.invoke(lc_messages)
print("ответ:", final_lc.content)

Что стоит заметить:

- `tool(get_room_info)` строит описание функции сам — по сигнатуре и докстрингу. Словарь `FUNCTION`
  писать не нужно.
- `AIMessage.tool_calls` — список словарей `{"name", "args", "id", "type"}`; `args` — уже словарь.
  Это одинаково для любого провайдера.
- Раскройте полный `AIMessage`. В `additional_kwargs` лежит `function_call` — родной формат GigaChat
  v1, и `functions_state_id`. Адаптер получил ответ в таком виде и сам превратил его в стандартный
  `tool_calls`. Идентификатор в `tool_calls[].id` сгенерировал адаптер.
- `usage_metadata` — расход токенов в общем для LangChain виде: `input_tokens`, `output_tokens` и
  `input_token_details.cache_read`. Для GigaChat `input_tokens` берётся из `prompt_tokens`, а
  `cache_read` — из `precached_prompt_tokens`, поэтому закэшированные токены во `input_tokens` не
  входят. Своего параметра для `X-Session-ID` у адаптера нет; кэш включается той же переменной SDK
  `gigachat.session_id_cvar`, что в разделе 4, — она действует и на вызовы через LangChain.

### 14.2. Четыре формата в одной таблице

Соберём из уже полученных ответов, как один и тот же вызов представлен в каждом формате. Новых
запросов ячейка не делает.

In [ ]:
comparison = [
    ["Chat Completions", assistant_message.tool_calls[0].function.name,
     assistant_message.tool_calls[0].function.arguments, assistant_message.tool_calls[0].id],
    ["Responses", responses_calls[0].name, responses_calls[0].arguments, responses_calls[0].call_id],
    ["Messages", tool_uses[0].name, tool_uses[0].input, tool_uses[0].id],
    ["LangChain", ai_message.tool_calls[0]["name"], ai_message.tool_calls[0]["args"], ai_message.tool_calls[0]["id"]],
]
rows = "".join("<tr>" + "".join(f"<td><code>{escape(str(value))}</code></td>" for value in row) + "</tr>"
               for row in comparison)
display(HTML("<table><tr><th>Формат</th><th>Имя</th><th>Аргументы</th><th>ID</th></tr>" + rows + "</table>"))

Имя и смысл аргументов совпадают. В Chat Completions и Responses аргументы — строка с JSON,
в Messages и LangChain — словарь. Идентификаторы разные, потому что это четыре отдельных вызова.

### 14.3. Что упростилось и что осталось задачей приложения

LangChain дал общие объекты сообщений и вызовов, описание функции строится само, код стал короче.
При этом функцию по-прежнему исполняет приложение, историю хранит приложение, и следующий запрос
делает тоже приложение. Особенности провайдера остаются: доступные параметры, поддержка инструментов,
способ учёта токенов. Общий слой упрощает замену модели, но эти различия всё равно приходится
проверять.

## 15. Практика

### 15.1. Новый факт при прежнем вопросе

Предположим, в аудиторию A-101 поставили дополнительные столы, и мест стало 32. Проверим, что модель
ответит по новым данным, хотя вопрос и инструкция прежние.

Задание:

1. Сохраните исходную вместимость A-101 и установите `32`.
2. Выполните `get_room_info` через `execute_tool` с аргументами из `ai_message.tool_calls`.
3. Создайте `ToolMessage` с новым результатом и идентификатором из того же вызова.
4. Вызовите `llm` с `SystemMessage`, вопросом, `ai_message` и новым результатом.
5. Верните исходную вместимость — даже если запрос к модели завершится ошибкой.

Критерии: функция вернула 32; идентификатор результата совпадает с идентификатором вызова; модель
ответила с учётом новой вместимости. Формулировка ответа может быть любой.

In [ ]:
# TODO: сохраните исходную вместимость A-101 и установите 32.
# TODO: выполните get_room_info через execute_tool с аргументами из ai_message.tool_calls.
# TODO: создайте ToolMessage с новым результатом и исходным ID вызова.
# TODO: вызовите llm с SystemMessage(content=SYSTEM), вопросом, ai_message и новым результатом.
# TODO: восстановите исходную вместимость (в том числе при ошибке API).

<details><summary>Подсказка и решение — после собственной попытки</summary>

Для пятого шага подходит `try` / `finally`: данные меняются внутри `try`, а восстанавливаются в
`finally` — при любом исходе. Если что-то пошло не так, ячейка «Сброс учебного справочника» в
разделе 7.2 возвращает исходные данные.

Готовое решение — функция `solve` в файле [solutions/room_capacity.py](solutions/room_capacity.py):

```python
from solutions.room_capacity import solve
exercise_reply = solve(llm, ai_message, QUESTION, SYSTEM)
print(exercise_reply.content)
```

</details>

Изменился только факт, который приложение передало модели, — и вместе с ним ответ. Данные, которые
приходят из инструментов, попадают прямо в контекст модели и определяют ответ. Поэтому важно, откуда
они берутся: если инструмент читает внешние источники, например веб-страницы, в данных могут
оказаться посторонние инструкции. Этот риск разберём на следующих занятиях.

### 15.2. Для самостоятельной работы

1. Повторите упражнение 15.1 в формате Responses или Messages.
2. Сравните тела HTTP-запросов до и после добавления результата функции. Какие поля относятся
   к контракту API, а какие — к данным об аудитории?
3. Передайте в `execute_tool` неизвестное имя функции или лишний аргумент. Объясните, почему
   приложение отклоняет такой вызов.
4. Запустите mitmproxy и найдите на участке Б запрос, соответствующий вашему обмену. Где в нём
   результат функции и почему в нём нет идентификатора вызова?
5. Повторите опыт с кэшем из раздела 4 для своего длинного текста и посчитайте, сколько токенов
   сэкономил `X-Session-ID`.

## 16. Итоги

### Что запомнить

- Модель ничего не знает о данных приложения. Доступ к ним даёт вызов функции: модель просит,
  приложение проверяет и исполняет, модель отвечает с учётом результата.
- Форматы API по-разному упаковывают одни и те же четыре шага: описание, вызов, исполнение, результат.
- Функцию исполняет приложение, и оно же обязано проверять имя и аргументы.
- В наших примерах историю передаёт приложение: каждый запрос несёт весь нужный контекст.
- `usage` — источник данных о расходе. Закэшированные токены не тарифицируются; кэш работает
  при одинаковом `X-Session-ID`.
- Ответы модели не воспроизводятся. Проверки строятся на структуре ответа и данных в нём.
- Прокси переводит форму запросов; модель остаётся той же. LangChain даёт общие объекты,
  а особенности провайдера остаются.

### Самопроверка

<details><summary>1. Чем ключ авторизации отличается от access token и кто обновляет токен?</summary>

Ключ — долговременный, по нему получают токен. Токен — временный пропуск на 30 минут, им подписан
каждый запрос к модели. SDK получает и обновляет токен сам.
</details>

<details><summary>2. Почему код, читающий ответ v1, не работает с ответом v2?</summary>

В v1 текст лежит в `choices[0].message.content`, а в v2 поля `choices` нет: ответ — список
`messages`, содержимое каждого сообщения — список частей.
</details>

<details><summary>3. Где в каждом формате лежит вызов функции и как с ним связывается результат?</summary>

GigaChat v2 — часть `function_call` в сообщении, результат — часть `function_result` с ролью `tool`.
Chat Completions — `tool_calls` и сообщение `tool` с `tool_call_id`. Responses — элементы
`function_call` и `function_call_output`, связь по `call_id`. Messages — блоки `tool_use` и
`tool_result`, связь по `tool_use_id`.
</details>

<details><summary>4. Почему смена <code>base_url</code> не превращает GigaChat в модель OpenAI?</summary>

`base_url` меняет адрес, на который библиотека отправляет запрос. Прокси по этому адресу переводит
формат и передаёт запрос в GigaChat. Отвечает GigaChat.
</details>

<details><summary>5. Что нормализует LangChain и что остаётся задачей приложения?</summary>

LangChain даёт общие сообщения и вызовы (`name`, словарь `args`, `id`) и строит описание функции.
Исполнение функции, хранение истории, проверка аргументов и особенности провайдера остаются
за приложением.
</details>

### Диагностика

| Симптом | Что проверить |
|---|---|
| Нет переменных `GIGACHAT_*` | файл `.env` в папке занятия; Jupyter запущен из неё; после правки `.env` перезапущено ядро |
| `401` от GigaChat | ключ авторизации и `scope`; новый ключ — в личном кабинете |
| Ошибка сертификата | в `.env` задано `GIGACHAT_VERIFY_SSL_CERTS=False`; ядро и прокси перезапущены |
| `Connection refused` к `127.0.0.1:8090` | запущен ли `tools/run_proxy.py` во втором терминале |
| `404` от прокси | путь в строке `HTTP →`; для Anthropic `base_url` без `/v1` |
| Нет вызова функции | реальный ответ модели и причина завершения |
| Ответ обрезан | `finish_reason: "length"` — увеличить `max_tokens`; обрезанные аргументы не исполнять |
| Ответ отличается от соседа | это нормально: сравнивайте структуру и данные |

Проверить окружение целиком: `uv run tools/check_env.py`.

### Дальше

На следующем занятии цикл «вызов → исполнение → результат» передадим агенту:
[агенты LangChain](https://docs.langchain.com/oss/python/langchain/agents), middleware и Deep Agents.
Остальные ссылки — в [SOURCES.md](SOURCES.md), подробности про перехват запросов — в
[HTTP_OBSERVATION.md](HTTP_OBSERVATION.md).

### Завершение

Ячейка ниже закрывает клиентов. Прокси и mitmproxy останавливаются в своих терминалах (`Ctrl+C`).
Перед тем как делиться ноутбуком, очистите выводы: в них тексты запросов и ответов.

In [ ]:
sdk.close()
openai_client.close()
anthropic_client.close()

## Приложение. Как найти преобразования в адаптере LangChain

Чтение для тех, кто хочет разобраться глубже. В `langchain-gigachat` 0.5.1 метод `_generate` строит
запрос, вызывает SDK (`.chat(payload)`, контракт v1) и превращает ответ в объекты LangChain.

В [исходнике адаптера](https://github.com/ai-forever/langchain-gigachat/blob/b0c2ed0a18da778d6ec987e5280eb8b500d1e2bf/libs/gigachat/langchain_gigachat/chat_models/gigachat.py)
найдите `_build_payload`, `_convert_message_to_dict`, `_generate` и `_create_chat_result`.
Проследите путь `ToolMessage` до сообщения API и путь ответа обратно в `AIMessage`.

Код установленной версии можно посмотреть так:

```python
import inspect
print(inspect.getsource(langchain_gigachat.GigaChat._generate))
```

Это чтение исходника для понимания. Прикладной код пользуется публичными методами вроде `invoke`.